# Better MTS-Dialog Dialogue-to-Note Fine-Tuning Colab

This is a more stable and stronger Colab pipeline for:

```text
doctor-patient dialogue → clinical note section
```

Compared with the previous notebook, this version:

- disables `fp16` to avoid `nan` loss on Colab T4
- does **not** compute ROUGE during training, avoiding tokenizer decoding crashes
- checks that labels/targets are real before training
- checks one forward-pass loss before training
- uses a stronger WangLab-style prompt and target format
- uses better hyperparameters: `lr=1e-4`, `warmup_ratio=0.1`, `weight_decay=0.01`, `label_smoothing=0.1`
- saves/resumes checkpoints from Drive
- evaluates with generation **after** training
- includes simple inference for your own dialogues

Recommended Google Drive layout:

```text
MyDrive/
  mts_dialog_finetuning/
    data/
      mts_dialog/
        training_set.csv
        validation_set.csv
```

Supported dataset columns:

```text
dialogue,note
```

or MTS-Dialog style:

```text
dialogue,section_header,section_text
```

## 1. Install dependencies

In [1]:
# IMPORTANT:
# We pin pandas to Colab's expected version to avoid dependency conflicts.
# After running this cell once, Runtime > Restart runtime if Colab asks or if pandas was previously upgraded.

!pip -q install -U \
  "transformers>=4.45.0,<5.0.0" \
  "datasets>=3.0.0" \
  "evaluate" \
  "rouge_score" \
  "sentencepiece" \
  "accelerate" \
  "sacrebleu" \
  "pandas==2.2.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.2 MB/s eta 0:00:00


## 2. Imports and Drive mount

In [2]:
import os
import re
import json
import math
import random
import inspect
import dataclasses
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped or unavailable:", repr(e))

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Main config

In [3]:
# =========================
# EDIT THIS CELL IF NEEDED
# =========================

BASE_DIR = "/content/drive/MyDrive/mts_dialog_finetuning"
DATA_DIR = f"{BASE_DIR}/data/mts_dialog"

CONFIG = {
    # Data paths
    "base_dir": BASE_DIR,
    "data_dir": DATA_DIR,
    "train_file": f"{DATA_DIR}/training_set.csv",
    "validation_file": f"{DATA_DIR}/validation_set.csv",

    # Model
    # Start with base. Try large later if you have enough GPU memory.
    "model_name": "google/flan-t5-base",

    # Output paths
    "processed_dataset_dir": f"{BASE_DIR}/processed_better/mts_dialog_processed",
    "tokenized_dataset_dir": f"{BASE_DIR}/processed_better/mts_dialog_tokenized",
    "model_output_dir": f"{BASE_DIR}/models_better/dialogue-to-note-flan-t5-base",
    "predictions_dir": f"{BASE_DIR}/predictions_better",

    # Sequence lengths
    "max_input_length": 1024,
    "max_target_length": 512,

    # Stronger hyperparameters
    "num_train_epochs": 20,
    "learning_rate": 1e-4,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "label_smoothing_factor": 0.1,

    # Colab-safe batch setup
    # Effective batch size = per_device_train_batch_size * gradient_accumulation_steps
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,

    # Generation/eval
    "generation_num_beams": 2,

    # Stability
    # Keep fp16 False on Colab T4 because your previous run had nan validation loss.
    "fp16": False,
    "gradient_checkpointing": True,

    # Reproducibility
    "seed": 42,

    # Debugging switches
    # Set to True for a quick smoke test before full training.
    "smoke_test": False,
    "smoke_train_examples": 64,
    "smoke_validation_examples": 32,

    # Checkpoint behavior
    "resume_from_checkpoint_if_available": True,
}

for key in [
    "base_dir",
    "data_dir",
    "processed_dataset_dir",
    "tokenized_dataset_dir",
    "model_output_dir",
    "predictions_dir",
]:
    Path(CONFIG[key]).mkdir(parents=True, exist_ok=True)

CONFIG_PATH = f"{BASE_DIR}/better_config.json"
with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Saved config to:", CONFIG_PATH)
print(json.dumps(CONFIG, indent=2))

Saved config to: /content/drive/MyDrive/mts_dialog_finetuning/better_config.json
{
  "base_dir": "/content/drive/MyDrive/mts_dialog_finetuning",
  "data_dir": "/content/drive/MyDrive/mts_dialog_finetuning/data/mts_dialog",
  "train_file": "/content/drive/MyDrive/mts_dialog_finetuning/data/mts_dialog/training_set.csv",
  "validation_file": "/content/drive/MyDrive/mts_dialog_finetuning/data/mts_dialog/validation_set.csv",
  "model_name": "google/flan-t5-base",
  "processed_dataset_dir": "/content/drive/MyDrive/mts_dialog_finetuning/processed_better/mts_dialog_processed",
  "tokenized_dataset_dir": "/content/drive/MyDrive/mts_dialog_finetuning/processed_better/mts_dialog_tokenized",
  "model_output_dir": "/content/drive/MyDrive/mts_dialog_finetuning/models_better/dialogue-to-note-flan-t5-base",
  "predictions_dir": "/content/drive/MyDrive/mts_dialog_finetuning/predictions_better",
  "max_input_length": 1024,
  "max_target_length": 512,
  "num_train_epochs": 20,
  "learning_rate": 0.0001,


## 4. Check dataset files

In [4]:
train_file = Path(CONFIG["train_file"])
val_file = Path(CONFIG["validation_file"])

print("Train file exists:", train_file.exists(), train_file)
print("Validation file exists:", val_file.exists(), val_file)

if not train_file.exists() or not val_file.exists():
    raise FileNotFoundError(
        "Dataset files not found. Upload them to Drive or edit CONFIG['train_file'] and CONFIG['validation_file']."
    )

train_df = pd.read_csv(train_file)
val_df = pd.read_csv(val_file)

print("\nTrain shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTrain columns:")
print(list(train_df.columns))

print("\nFirst train row:")
display(train_df.head(1))

Train file exists: True /content/drive/MyDrive/mts_dialog_finetuning/data/mts_dialog/training_set.csv
Validation file exists: True /content/drive/MyDrive/mts_dialog_finetuning/data/mts_dialog/validation_set.csv

Train shape: (1201, 4)
Validation shape: (100, 4)

Train columns:
['ID', 'section_header', 'section_text', 'dialogue']

First train row:


,ID,section_header,section_text,dialogue
0,0,GENHX,The patient is a 76-year-old white female who ...,Doctor: What brings you back into the clinic t...


## 5. Load raw dataset

In [5]:
from datasets import load_dataset, DatasetDict

raw_dataset = load_dataset(
    "csv",
    data_files={
        "train": CONFIG["train_file"],
        "validation": CONFIG["validation_file"],
    },
)

print(raw_dataset)
print("Train columns:", raw_dataset["train"].column_names)
print("Validation columns:", raw_dataset["validation"].column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['ID', 'section_header', 'section_text', 'dialogue'],
        num_rows: 1201
    })
    validation: Dataset({
        features: ['ID', 'section_header', 'section_text', 'dialogue'],
        num_rows: 100
    })
})
Train columns: ['ID', 'section_header', 'section_text', 'dialogue']
Validation columns: ['ID', 'section_header', 'section_text', 'dialogue']


## 6. Prepare target format

This version uses a more explicit target:

```text
Section header: ...
Section text: ...
```

For MTS-Dialog, this helps because the task is not just generic summarization. The model should learn the clinical note section too.

In [6]:
def clean_text(value):
    if value is None:
        return ""
    value = str(value)
    value = value.replace("\r\n", "\n").replace("\r", "\n")
    value = re.sub(r"[ \t]+", " ", value)
    value = re.sub(r"\n{3,}", "\n\n", value)
    return value.strip()


def build_target(example):
    # Supports:
    # 1. Custom CSV columns: dialogue, note
    # 2. MTS-Dialog columns: dialogue, section_header, section_text
    # 3. Fallback: dialogue, section_text

    if "section_header" in example and "section_text" in example:
        header = clean_text(example.get("section_header", ""))
        section_text = clean_text(example.get("section_text", ""))

        if header and section_text:
            return f"Section header: {header}\nSection text: {section_text}"

        if section_text:
            return f"Section text: {section_text}"

    if "note" in example:
        note = clean_text(example.get("note", ""))
        if note:
            return f"Section text: {note}"

    if "section_text" in example:
        section_text = clean_text(example.get("section_text", ""))
        if section_text:
            return f"Section text: {section_text}"

    return ""


PROMPT_PREFIX = (
    "Summarize the following patient-doctor dialogue into a clinical note section. "
    "Include all medically relevant information. "
    "First predict the most relevant clinical note section header, then write the section text. "
    "Use this exact output format: 'Section header: ... Section text: ...'.\n\n"
    "Dialogue:\n"
)


def prepare_example(example):
    dialogue = clean_text(example.get("dialogue", ""))
    target_note = build_target(example)

    return {
        "dialogue": dialogue,
        "input_text": PROMPT_PREFIX + dialogue,
        "target_note": target_note,
    }


def keep_valid(example):
    return bool(example["dialogue"]) and bool(example["target_note"])


processed = raw_dataset.map(
    prepare_example,
    remove_columns=raw_dataset["train"].column_names,
)

processed = processed.filter(keep_valid)

if CONFIG["smoke_test"]:
    processed = DatasetDict({
        "train": processed["train"].select(range(min(CONFIG["smoke_train_examples"], len(processed["train"])))),
        "validation": processed["validation"].select(range(min(CONFIG["smoke_validation_examples"], len(processed["validation"])))),
    })

print(processed)

print("\nSample input:")
print(processed["train"][0]["input_text"][:1500])

print("\nSample target:")
print(processed["train"][0]["target_note"][:1500])

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1201 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'input_text', 'target_note'],
        num_rows: 1201
    })
    validation: Dataset({
        features: ['dialogue', 'input_text', 'target_note'],
        num_rows: 100
    })
})

Sample input:
Summarize the following patient-doctor dialogue into a clinical note section. Include all medically relevant information. First predict the most relevant clinical note section header, then write the section text. Use this exact output format: 'Section header: ... Section text: ...'.

Dialogue:
Doctor: What brings you back into the clinic today, miss? 
Patient: I came in for a refill of my blood pressure medicine. 
Doctor: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones. Have you noticed any changes or do you have any concerns regarding these issues? 
Patient: No. 
Doctor: Have you had any fever or chills, cough,

## 7. Dataset sanity checks

In [7]:
# Check for empty / suspicious examples.
for split in ["train", "validation"]:
    df = processed[split].to_pandas()
    df["dialogue_chars"] = df["dialogue"].str.len()
    df["target_chars"] = df["target_note"].str.len()

    print("\n" + split.upper())
    print("Rows:", len(df))
    print("Empty dialogues:", (df["dialogue_chars"] == 0).sum())
    print("Empty targets:", (df["target_chars"] == 0).sum())
    display(df[["dialogue_chars", "target_chars"]].describe())

    print("\nShortest targets:")
    display(df.sort_values("target_chars").head(5)[["target_note", "target_chars"]])

processed.save_to_disk(CONFIG["processed_dataset_dir"])
print("\nSaved processed dataset to:", CONFIG["processed_dataset_dir"])


TRAIN
Rows: 1201
Empty dialogues: 0
Empty targets: 0


,dialogue_chars,target_chars
count,1201.000000,1201.000000
mean,595.608659,280.124063
std,659.876348,389.219604
min,41.000000,39.000000
25%,194.000000,73.000000
50%,356.000000,127.000000
75%,749.000000,316.000000
max,8861.000000,6224.000000



Shortest targets:


,target_note,target_chars
756,Section header: CC\nSection text: Fever.,39
252,Section header: CC\nSection text: Nausea.,40
769,Section header: CC\nSection text: Urology.,41
330,Section header: CC\nSection text: Syncope.,41
301,Section header: CC\nSection text: Surgery.,41



VALIDATION
Rows: 100
Empty dialogues: 0
Empty targets: 0


...


Shortest targets:


,target_note,target_chars
70,Section header: ALLERGY\nSection text: None.,43
57,Section header: MEDICATIONS\nSection text: None.,47
80,Section header: PASTSURGICAL\nSection text: None.,48
85,Section header: FAM/SOCHX\nSection text: unknown.,48
40,Section header: DISPOSITION\nSection text: To ...,50


Saving the dataset (0/1 shards):   0%|          | 0/1201 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]


Saved processed dataset to: /content/drive/MyDrive/mts_dialog_finetuning/processed_better/mts_dialog_processed


## 8. Load model and tokenizer

In [8]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

set_seed(int(CONFIG["seed"]))

MODEL_NAME = CONFIG["model_name"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

if CONFIG["gradient_checkpointing"]:
    model.gradient_checkpointing_enable()
    # Important for gradient checkpointing with seq2seq generation models.
    model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Loaded:", MODEL_NAME)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded: google/flan-t5-base
Total parameters: 247,577,856
Trainable parameters: 247,577,856


## 9. Tokenize dataset

In [9]:
MAX_INPUT_LENGTH = int(CONFIG["max_input_length"])
MAX_TARGET_LENGTH = int(CONFIG["max_target_length"])

def preprocess(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target_note"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = processed.map(
    preprocess,
    batched=True,
    remove_columns=processed["train"].column_names,
)

tokenized.save_to_disk(CONFIG["tokenized_dataset_dir"])

print(tokenized)
print("Saved tokenized dataset to:", CONFIG["tokenized_dataset_dir"])

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1201 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1201
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})
Saved tokenized dataset to: /content/drive/MyDrive/mts_dialog_finetuning/processed_better/mts_dialog_tokenized


## 10. Decode sanity check

In [10]:
sample = tokenized["train"][0]

print("Input IDs length:", len(sample["input_ids"]))
print("Labels length:", len(sample["labels"]))
print("First 30 label IDs:", sample["labels"][:30])

print("\nDecoded input:")
print(tokenizer.decode(sample["input_ids"][:300], skip_special_tokens=True))

print("\nDecoded target:")
print(tokenizer.decode(sample["labels"][:300], skip_special_tokens=True))

# Hard assertion: target must not be empty.
decoded_target = tokenizer.decode(sample["labels"], skip_special_tokens=True).strip()
assert decoded_target, "Decoded target is empty. Something is wrong with target preparation."

Input IDs length: 229
Labels length: 119
First 30 label IDs: [5568, 13956, 10, 3, 18464, 566, 4, 5568, 1499, 10, 37, 1868, 19, 3, 9, 3, 3959, 18, 1201, 18, 1490, 872, 3955, 113, 6621, 12, 8, 5998, 469, 5330]

Decoded input:
Summarize the following patient-doctor dialogue into a clinical note section. Include all medically relevant information. First predict the most relevant clinical note section header, then write the section text. Use this exact output format: 'Section header: ... Section text: ...'. Dialogue: Doctor: What brings you back into the clinic today, miss? Patient: I came in for a refill of my blood pressure medicine. Doctor: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones. Have you noticed any changes or do you have any concerns regarding these issues? Patient: No. Doctor: Have you had any fever or chills, cough, congestion, nausea, vomiting, chest pain

## 11. One-batch loss sanity check

Before full training, this checks whether the model can compute a finite, positive loss.

If this prints `nan`, `inf`, or exactly `0.0`, do not train yet.

In [11]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

batch_examples = [tokenized["train"][i] for i in range(min(2, len(tokenized["train"])))]
batch = data_collator(batch_examples)
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)
    sanity_loss = outputs.loss

print("Sanity loss:", sanity_loss.item())

assert torch.isfinite(sanity_loss), "Loss is not finite. Check data, labels, or precision settings."
assert sanity_loss.item() > 0, "Loss is zero. This is suspicious; check label preparation."

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Sanity loss: 2.685407876968384


## 12. Trainer setup

Important difference from the previous notebook:

- `predict_with_generate=False` during training
- no `compute_metrics` during training
- we evaluate generation separately after training

This avoids the decoding crash and keeps training focused on stable loss.

In [12]:
# Version-tolerant args: some versions use eval_strategy, older versions use evaluation_strategy.
arg_field_names = {field.name for field in dataclasses.fields(Seq2SeqTrainingArguments)}

training_kwargs = {
    "output_dir": CONFIG["model_output_dir"],
    "save_strategy": "epoch",

    "learning_rate": float(CONFIG["learning_rate"]),
    "warmup_ratio": float(CONFIG["warmup_ratio"]),
    "weight_decay": float(CONFIG["weight_decay"]),
    "label_smoothing_factor": float(CONFIG["label_smoothing_factor"]),

    "per_device_train_batch_size": int(CONFIG["per_device_train_batch_size"]),
    "per_device_eval_batch_size": int(CONFIG["per_device_eval_batch_size"]),
    "gradient_accumulation_steps": int(CONFIG["gradient_accumulation_steps"]),

    "dataloader_pin_memory": False,
    "num_train_epochs": float(CONFIG["num_train_epochs"]),

    # Stable training: evaluate loss only during train.
    "predict_with_generate": False,

    "logging_steps": 10,
    "save_total_limit": 3,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,

    # Stability: keep fp16 off for this project on Colab T4.
    "fp16": bool(CONFIG["fp16"]),

    "report_to": "none",
    "seed": int(CONFIG["seed"]),
}

if "eval_strategy" in arg_field_names:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**training_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized["train"],
    "eval_dataset": tokenized["validation"],
    "data_collator": data_collator,
}

trainer_signature = inspect.signature(Seq2SeqTrainer.__init__)
if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Trainer ready.")
print("Output dir:", CONFIG["model_output_dir"])
print("Effective batch size:", CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"])

Trainer ready.
Output dir: /content/drive/MyDrive/mts_dialog_finetuning/models_better/dialogue-to-note-flan-t5-base
Effective batch size: 8


## 13. Check for existing checkpoints

In [13]:
def get_latest_checkpoint(output_dir):
    output_dir = Path(output_dir)
    if not output_dir.exists():
        return None

    checkpoints = []
    for path in output_dir.glob("checkpoint-*"):
        if path.is_dir():
            try:
                step = int(path.name.split("-")[-1])
                checkpoints.append((step, str(path)))
            except ValueError:
                pass

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])
    return checkpoints[-1][1]


latest_checkpoint = get_latest_checkpoint(CONFIG["model_output_dir"])

print("Latest checkpoint:", latest_checkpoint)

if latest_checkpoint and CONFIG["resume_from_checkpoint_if_available"]:
    print("Training will resume from this checkpoint.")
else:
    print("Training will start from the base model.")

Latest checkpoint: None
Training will start from the base model.


## 14. Train

In [14]:
resume_checkpoint = None

if CONFIG["resume_from_checkpoint_if_available"]:
    resume_checkpoint = get_latest_checkpoint(CONFIG["model_output_dir"])

train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

print("\nTraining done.")
print(train_result)

Epoch,Training Loss,Validation Loss
1,3.320300,3.235074
2,3.102000,2.980635
3,2.941200,2.880409
4,2.834400,2.824766
5,2.751100,2.803403
6,2.525100,2.787708
7,2.495800,2.775794
8,2.484700,2.783665
9,2.426900,2.767660
10,2.280100,2.772007


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



Training done.
TrainOutput(global_step=3020, training_loss=2.5125651669028577, metrics={'train_runtime': 5038.3821, 'train_samples_per_second': 4.767, 'train_steps_per_second': 0.599, 'total_flos': 9342250215367680.0, 'train_loss': 2.5125651669028577, 'epoch': 20.0})


## 15. Save final model

In [15]:
trainer.save_model(CONFIG["model_output_dir"])
tokenizer.save_pretrained(CONFIG["model_output_dir"])

# Re-enable cache for faster generation after training.
model.config.use_cache = True

print("Saved model to:", CONFIG["model_output_dir"])

Saved model to: /content/drive/MyDrive/mts_dialog_finetuning/models_better/dialogue-to-note-flan-t5-base


## 16. Final eval loss

In [16]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 2.767659902572632, 'eval_runtime': 4.7708, 'eval_samples_per_second': 20.961, 'eval_steps_per_second': 10.48, 'epoch': 20.0}


## 17. Generation functions

Now we evaluate generated text after training, separately from the training loop.

In [17]:
import evaluate

rouge = evaluate.load("rouge")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def generate_note(dialogue, max_input_length=None, max_new_tokens=None, num_beams=None):
    max_input_length = max_input_length or int(CONFIG["max_input_length"])
    max_new_tokens = max_new_tokens or int(CONFIG["max_target_length"])
    num_beams = num_beams or int(CONFIG["generation_num_beams"])

    dialogue = clean_text(dialogue)
    prompt = PROMPT_PREFIX + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


def evaluate_generation(max_examples=50):
    validation = processed["validation"]

    if max_examples is not None:
        validation = validation.select(range(min(max_examples, len(validation))))

    rows = []

    for i, example in enumerate(validation):
        generated = generate_note(example["dialogue"])

        rows.append({
            "index": i,
            "dialogue": example["dialogue"],
            "reference_note": example["target_note"],
            "generated_note": generated,
        })

        if (i + 1) % 10 == 0:
            print(f"Generated {i + 1}/{len(validation)}")

    pred_df = pd.DataFrame(rows)

    scores = rouge.compute(
        predictions=pred_df["generated_note"].tolist(),
        references=pred_df["reference_note"].tolist(),
        use_stemmer=True,
    )

    return pred_df, scores

## 18. Evaluate generated notes on validation subset

In [18]:
# Start with 50 examples. Set MAX_GENERATION_EVAL_EXAMPLES = None for full validation, but it will be slower.
MAX_GENERATION_EVAL_EXAMPLES = 50

pred_df, generation_scores = evaluate_generation(max_examples=MAX_GENERATION_EVAL_EXAMPLES)

print(generation_scores)
display(pred_df.head())

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pred_path = f'{CONFIG["predictions_dir"]}/better_validation_predictions_{timestamp}.csv'
pred_df.to_csv(pred_path, index=False)

print("Saved predictions to:", pred_path)

Generated 10/50
Generated 20/50
Generated 30/50
Generated 40/50
Generated 50/50
{'rouge1': np.float64(0.5704860363333735), 'rouge2': np.float64(0.36700418012002894), 'rougeL': np.float64(0.5027738582040766), 'rougeLsum': np.float64(0.5012025499555934)}


,index,dialogue,reference_note,generated_note
0,0,Doctor: When did your pain begin? \nPatient: I...,Section header: GENHX\nSection text: The patie...,Section header: GENHX Section text: The patien...
1,1,"Doctor: Hey, bud. What brings you in today? \n...",Section header: ROS\nSection text: As mentione...,Section header: CC Section text: Rashes of upp...
2,2,Doctor: Has anything changed in your medical h...,Section header: PASTMEDICALHX\nSection text: E...,Section header: PASTMEDICALHX Section text: No...
3,3,Doctor: How've you been treating your acne? \n...,Section header: MEDICATIONS\nSection text: Acc...,Section header: MEDICATIONS Section text: Accu...
4,4,Doctor: Have you been experiencing any mental ...,Section header: CC\nSection text: Confusion an...,Section header: ROS Section text: DISPOSITION:...


Saved predictions to: /content/drive/MyDrive/mts_dialog_finetuning/predictions_better/better_validation_predictions_20260516_222317.csv


## 19. Inspect a few predictions

In [19]:
for i in range(min(5, len(pred_df))):
    print("=" * 120)
    print("DIALOGUE")
    print(pred_df.loc[i, "dialogue"][:2000])

    print("\nREFERENCE")
    print(pred_df.loc[i, "reference_note"][:1200])

    print("\nGENERATED")
    print(pred_df.loc[i, "generated_note"][:1200])

DIALOGUE
Doctor: When did your pain begin? 
Patient: I've had low back pain for about eight years now.
Doctor: Is there any injury? 
 Patient: Yeah, it started when I fell in an A B C store.
Doctor: How old are you now?
Patient: I'm twenty six. 
Doctor: What kind of treatments have you had for this low back pain? 
Patient: Yeah, I got referred to P T, and I went, but only once or twice, um, and if I remember right, they only did the electrical stimulation, and heat. 
Doctor: I see, how has your pain progressed over the last eight years? 
Patient: It's been pretty continuous, but it's been at varying degrees, sometimes are better than others. 
Doctor: Do you have any children? 
Patient: Yes, I had my son in August of two thousand eight, and I've had back pain since giving birth. 
Doctor: Have you had any falls since the initial one? 
Patient: Yes, I fell four or five days ago while I was mopping the floor. 
Doctor: Did you land on your lower back again?
Patient: Yes, right onto my tailb

## 20. Generate a note from your own dialogue

In [20]:
my_dialogue = """
Doctor: What brings you in today?
Patient: I've had a cough and fever for three days.
Doctor: Any shortness of breath?
Patient: A little when I walk upstairs.
Doctor: Any chest pain?
Patient: No chest pain.
Doctor: Are you taking anything?
Patient: Just paracetamol.
"""

print(generate_note(my_dialogue))

Section header: CC Section text: Cough and fever for 3 days. No shortness of breath. No chest pain. The patient is on paracetamol.


## 21. Optional interactive textbox

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    dialogue_box = widgets.Textarea(
        value="Doctor: ...\nPatient: ...",
        placeholder="Paste your doctor-patient dialogue here",
        description="Dialogue:",
        layout=widgets.Layout(width="100%", height="250px"),
    )

    button = widgets.Button(description="Generate note")
    output = widgets.Output()

    def on_click(_):
        with output:
            output.clear_output()
            print(generate_note(dialogue_box.value))

    button.on_click(on_click)
    display(dialogue_box, button, output)

except Exception as e:
    print("Interactive widget unavailable:", repr(e))

## 22. Optional: stronger model settings

Only use these after the base model pipeline works.

For `google/flan-t5-large`, free Colab may run out of memory with full fine-tuning.

Suggested changes:

```python
CONFIG["model_name"] = "google/flan-t5-large"
CONFIG["model_output_dir"] = f'{CONFIG["base_dir"]}/models_better/dialogue-to-note-flan-t5-large'
CONFIG["per_device_train_batch_size"] = 1
CONFIG["per_device_eval_batch_size"] = 1
CONFIG["gradient_accumulation_steps"] = 8
CONFIG["gradient_checkpointing"] = True
CONFIG["fp16"] = False
```

Then rerun from the model loading cell onward.

In [ ]:
# Uncomment this block to switch to FLAN-T5 large.

# CONFIG["model_name"] = "google/flan-t5-large"
# CONFIG["model_output_dir"] = f'{CONFIG["base_dir"]}/models_better/dialogue-to-note-flan-t5-large'
# CONFIG["per_device_train_batch_size"] = 1
# CONFIG["per_device_eval_batch_size"] = 1
# CONFIG["gradient_accumulation_steps"] = 8
# CONFIG["gradient_checkpointing"] = True
# CONFIG["fp16"] = False
#
# with open(CONFIG_PATH, "w") as f:
#     json.dump(CONFIG, f, indent=2)
#
# print(json.dumps(CONFIG, indent=2))

## 23. Quick troubleshooting

### If training is too slow

Set:

```python
CONFIG["num_train_epochs"] = 5
```

for testing, then go back to 20.

### If memory crashes

Use:

```python
CONFIG["per_device_train_batch_size"] = 1
CONFIG["gradient_accumulation_steps"] = 8
```

### If loss is `nan`

Keep:

```python
CONFIG["fp16"] = False
```

and rerun from the model loading cell.

### If training loss is exactly `0.0`

Do not trust the run. Rerun the decode sanity check and one-batch loss sanity check.